# 🔍 Fine-Tuning AI Image Detector

Este notebook permite hacer fine-tuning del modelo `umm-maybe/AI-image-detector` con tu propio dataset.

## Pasos:
1. Conectar GPU
2. Subir dataset
3. Entrenar modelo
4. Descargar modelo entrenado

## 1️⃣ Verificar GPU y Instalar Dependencias

In [ ]:
# Verificar GPU disponible
!nvidia-smi

In [ ]:
# Instalar dependencias
!pip install -q transformers datasets accelerate tensorboard scikit-learn pillow

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ Subir Dataset

Sube un archivo ZIP con esta estructura:
```
dataset.zip
├── train/
│   ├── real/     (imágenes reales)
│   └── ai/       (imágenes generadas por IA)
└── val/
    ├── real/
    └── ai/
```

In [ ]:
# Opción 1: Subir ZIP manualmente
from google.colab import files
print("Sube tu archivo dataset.zip:")
uploaded = files.upload()

In [ ]:
# Extraer dataset
!unzip -q dataset.zip -d /content/dataset
!ls -la /content/dataset/train/
!echo "---"
!echo "Imágenes en train/real:" $(ls /content/dataset/train/real | wc -l)
!echo "Imágenes en train/ai:" $(ls /content/dataset/train/ai | wc -l)

In [ ]:
# Opción 2: Usar Google Drive (alternativa)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dataset.zip /content/
# !unzip -q dataset.zip -d /content/dataset

## 3️⃣ Configurar Entrenamiento

In [ ]:
from transformers import (
    AutoModelForImageClassification,
    AutoImageProcessor,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, DatasetDict
from torchvision.transforms import (
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    ToTensor,
    Resize,
    CenterCrop,
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# Configuración
MODEL_NAME = "umm-maybe/AI-image-detector"  # Modelo base
DATASET_PATH = "/content/dataset"
OUTPUT_DIR = "/content/ai-detector-finetuned"

# Hiperparámetros
EPOCHS = 3
BATCH_SIZE = 16  # Reducir a 8 si hay errores de memoria
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

In [ ]:
# Cargar modelo y processor
print("Cargando modelo base...")
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True,
)
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

# Configurar labels
model.config.label2id = {"real": 0, "ai": 1}
model.config.id2label = {0: "real", 1: "ai"}

print(f"Modelo cargado: {MODEL_NAME}")
print(f"Parámetros: {model.num_parameters():,}")

In [ ]:
# Cargar dataset
print("Cargando dataset...")
dataset = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(dataset)

In [ ]:
# Definir transformaciones
image_mean = processor.image_mean
image_std = processor.image_std
size = processor.size["height"]

# Transformaciones para entrenamiento (con augmentation)
train_transforms = Compose([
    RandomResizedCrop(size),
    RandomHorizontalFlip(),
    ToTensor(),
    Normalize(mean=image_mean, std=image_std),
])

# Transformaciones para validación (sin augmentation)
val_transforms = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    Normalize(mean=image_mean, std=image_std),
])

def apply_train_transforms(examples):
    examples["pixel_values"] = [
        train_transforms(img.convert("RGB")) for img in examples["image"]
    ]
    return examples

def apply_val_transforms(examples):
    examples["pixel_values"] = [
        val_transforms(img.convert("RGB")) for img in examples["image"]
    ]
    return examples

In [ ]:
# Aplicar transformaciones
print("Aplicando transformaciones...")
train_dataset = dataset["train"].with_transform(apply_train_transforms)
val_dataset = dataset["validation"].with_transform(apply_val_transforms)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

In [ ]:
# Función para calcular métricas
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )
    
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Collator para batching
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

## 4️⃣ Entrenar Modelo

In [ ]:
# Configurar argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    remove_unused_columns=False,
    fp16=True,  # Mixed precision para más velocidad
    dataloader_num_workers=2,
    report_to="tensorboard",
)

# Crear Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

print("Configuración lista. Iniciando entrenamiento...")

In [ ]:
# ¡ENTRENAR!
trainer.train()

In [ ]:
# Evaluar modelo final
print("\n📊 Evaluación final:")
metrics = trainer.evaluate()
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

## 5️⃣ Guardar y Descargar Modelo

In [ ]:
# Guardar modelo
FINAL_MODEL_PATH = "/content/ai-detector-final"
trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)
print(f"Modelo guardado en: {FINAL_MODEL_PATH}")

In [ ]:
# Comprimir modelo para descargar
!zip -r /content/ai-detector-finetuned.zip /content/ai-detector-final
print("\n📦 Modelo comprimido. Descargando...")

# Descargar
from google.colab import files
files.download("/content/ai-detector-finetuned.zip")

## 6️⃣ Probar Modelo (Opcional)

In [ ]:
# Probar con una imagen
from transformers import pipeline
from PIL import Image

# Cargar modelo entrenado
classifier = pipeline(
    "image-classification",
    model=FINAL_MODEL_PATH,
    device=0,
)

# Subir imagen de prueba
print("Sube una imagen para probar:")
test_upload = files.upload()
test_image_path = list(test_upload.keys())[0]

# Clasificar
result = classifier(test_image_path)
print(f"\n🔍 Resultado: {result}")

## 📝 Notas

### Usar el modelo entrenado en tu API:

1. Descomprime `ai-detector-finetuned.zip`
2. Copia la carpeta a tu proyecto
3. Modifica `detector.py`:

```python
def __init__(self, model_name: str = "./models/ai-detector-final"):
```

### Subir a HuggingFace Hub:

```python
from huggingface_hub import login
login(token="tu-token")

trainer.push_to_hub("tu-usuario/ai-detector-custom")
```